# Fetch Spatial Features — Road Density & POI Count (Overpass API)

Calcula **road length** y **POI density** para cada una de las 247 zonas del dataset ST-EVCDP usando la API **Overpass de OpenStreetMap** (gratuita, sin API key).

- Fuente: [overpass-api.de](https://overpass-api.de)
- Input: coordenadas `lon`/`la` de `information.csv`
- Método: radio de **1 km** alrededor del centroide de cada zona
- Output: `spatial_features_shenzhen.csv`

| Feature | Paper cita (§3.1.1) | Cómo se calcula |
|---|---|---|
| **Road length (km)** | `"Road Length = 83.23 km"` | Suma longitudes de `way["highway"]` en radio 1 km |
| **POI count** | `"POI density"` | Cuenta nodos `amenity + shop + leisure` en radio 1 km |

- Rate limit: ~1 req/s → ~8 min para 247 zonas
- Checkpointing: guarda progreso parcial cada 20 zonas

In [12]:
!pip install tqdm requests -q

import requests
import pandas as pd
import numpy as np
import math
import time
import json
from pathlib import Path
from tqdm.notebook import tqdm

In [13]:
# ── CONFIGURA AQUÍ tu ruta a los datasets ────────────────────────────
# None = auto-descubrir en Drive (Colab/extensión Colab)
# Si falla, pon la ruta manualmente: "/content/drive/MyDrive/tu_carpeta"
DATASETS_PATH_OVERRIDE = None

import os, glob

if DATASETS_PATH_OVERRIDE:
    DATASETS_PATH = DATASETS_PATH_OVERRIDE
    print(f"Usando ruta manual: {DATASETS_PATH}")
else:
    candidates = glob.glob("/content/drive/**/*information.csv", recursive=True)
    if candidates:
        DATASETS_PATH = os.path.dirname(candidates[0])
        print(f"Encontrado automáticamente: {DATASETS_PATH}")
    else:
        try:
            from google.colab import drive
            if not os.path.isdir("/content/drive/MyDrive"):
                drive.mount("/content/drive")
            candidates = glob.glob("/content/drive/**/*information.csv", recursive=True)
            if candidates:
                DATASETS_PATH = os.path.dirname(candidates[0])
                print(f"Encontrado tras montar Drive: {DATASETS_PATH}")
            else:
                print("ERROR: information.csv no encontrado en Drive.")
                print("Pon la ruta correcta en DATASETS_PATH_OVERRIDE arriba.")
                DATASETS_PATH = None
        except ImportError:
            print("No estás en Colab. Pon la ruta en DATASETS_PATH_OVERRIDE.")
            DATASETS_PATH = None

if DATASETS_PATH:
    test_file = f"{DATASETS_PATH}/information.csv"
    if os.path.exists(test_file):
        csvs = glob.glob(f"{DATASETS_PATH}/*.csv")
        print(f"OK — {len(csvs)} CSVs encontrados:")
        for c in sorted(csvs):
            print(f"  {os.path.basename(c)}")
    else:
        print(f"ERROR: {test_file} no existe. Revisa DATASETS_PATH_OVERRIDE.")

Encontrado automáticamente: /content/drive/MyDrive/datasets
OK — 11 CSVs encontrados:
  addresses_shenzhen.csv
  adj.csv
  calendar_features.csv
  distance.csv
  duration.csv
  information.csv
  occupancy.csv
  price.csv
  stations.csv
  time.csv
  volume.csv


In [14]:
# ── Rutas ─────────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    import os
    if not os.path.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    IN_COLAB = True
    print("Drive montado.")
except ImportError:
    IN_COLAB = False
    # DATASETS_PATH se define en la celda de verificación de ruta
    if 'DATASETS_PATH' not in dir() or DATASETS_PATH is None:
        DATASETS_PATH = "./datasets"
    print("Local mode:", DATASETS_PATH)

CHECKPOINT_PATH = "spatial_checkpoint.json"
OUTPUT_PATH     = "spatial_features_shenzhen.csv"

# ── Cargar information.csv ────────────────────────────────────────────────
info = pd.read_csv(f"{DATASETS_PATH}/information.csv", index_col=0)
info.columns = [c.strip().lower().replace(" ", "_") for c in info.columns]

LAT_COL = "la"
LON_COL = "lon"

print(f"Zonas: {len(info)}")
print(f"Lat range: {info[LAT_COL].min():.4f} – {info[LAT_COL].max():.4f}")
print(f"Lon range: {info[LON_COL].min():.4f} – {info[LON_COL].max():.4f}")

Drive montado.
Zonas: 247
Lat range: 22.4862 – 22.7899
Lon range: 113.7904 – 114.5156


In [15]:
# ── Fórmula Haversine para longitud de segmentos de carretera ─────────────
def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Distancia en km entre dos coordenadas (fórmula haversine)."""
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))


def way_length_km(way: dict) -> float:
    """Calcula la longitud total (km) de una way de OSM a partir de su geometría."""
    geom = way.get("geometry", [])
    if len(geom) < 2:
        return 0.0
    total = 0.0
    for i in range(len(geom) - 1):
        total += haversine_km(geom[i]["lat"], geom[i]["lon"],
                              geom[i+1]["lat"], geom[i+1]["lon"])
    return total

In [16]:
# Múltiples endpoints Overpass con fallback
OVERPASS_ENDPOINTS = [
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass-api.de/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
]
RADIUS_M = 1000
OVERPASS_HEADERS = {
    "User-Agent": "ChatEV-Academic-Research/1.0 (jimena.milla@cloudlevante.com)",
    "Accept": "*/*",
}

HIGHWAY_TYPES = [
    "motorway", "trunk", "primary", "secondary", "tertiary",
    "residential", "unclassified", "motorway_link", "trunk_link",
    "primary_link", "secondary_link", "tertiary_link"
]

def build_roads_query(lat: float, lon: float, radius: int = RADIUS_M) -> str:
    hf = "|".join(HIGHWAY_TYPES)
    return f"""[out:json][timeout:90];
way["highway"~"{hf}"](around:{radius},{lat},{lon});
out body geom;"""

def build_poi_query(lat: float, lon: float, radius: int = RADIUS_M) -> str:
    return f"""[out:json][timeout:60];
(
  node["amenity"](around:{radius},{lat},{lon});
  node["shop"](around:{radius},{lat},{lon});
  node["leisure"](around:{radius},{lat},{lon});
);
out ids;"""

def query_overpass(query: str, retries: int = 3):
    """Prueba POST y GET en todos los endpoints. Devuelve JSON o None."""
    for attempt in range(retries):
        url = OVERPASS_ENDPOINTS[attempt % len(OVERPASS_ENDPOINTS)]
        try:
            # Intentar POST primero, luego GET si falla con 406
            resp = requests.post(url, data={"data": query},
                                 headers=OVERPASS_HEADERS, timeout=120)
            if resp.status_code == 406:
                # Algunos servidores prefieren GET
                resp = requests.get(url, params={"data": query},
                                    headers=OVERPASS_HEADERS, timeout=120)
            if resp.status_code == 429:
                wait = 30 * (attempt + 1)
                print(f"    Rate limit — {wait}s")
                time.sleep(wait)
                continue
            if resp.status_code in (502, 503, 504):
                print(f"    {resp.status_code} en {url[:35]}\u2026 reintentando")
                time.sleep(15)
                continue
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            print(f"    Error attempt {attempt+1} ({url[:35]}\u2026): {e}")
            time.sleep(8 * (attempt + 1))
    return None

def fetch_spatial(lat: float, lon: float) -> dict:
    """Dos queries separadas (roads + POIs) para reducir carga por petición."""
    r_data = query_overpass(build_roads_query(lat, lon))
    if r_data is None:
        return {"road_length_km": None, "poi_count": None, "radius_m": RADIUS_M, "error": True}
    road_km = sum(way_length_km(el) for el in r_data.get("elements", []) if el["type"] == "way")

    time.sleep(1.5)

    p_data = query_overpass(build_poi_query(lat, lon))
    if p_data is None:
        return {"road_length_km": None, "poi_count": None, "radius_m": RADIUS_M, "error": True}
    poi_count = sum(1 for el in p_data.get("elements", []) if el["type"] == "node")

    return {"road_length_km": round(road_km, 3), "poi_count": poi_count,
            "radius_m": RADIUS_M, "error": False}


# Test con zona 0
sample_lat = float(info[LAT_COL].iloc[0])
sample_lon = float(info[LON_COL].iloc[0])
print(f"Test zona 0: lat={sample_lat}, lon={sample_lon}")
result = fetch_spatial(sample_lat, sample_lon)
print(f"Resultado: {result}")

Test zona 0: lat=22.54041, lon=114.103
    Error attempt 1 (https://overpass.kumi.systems/api/i…): HTTPSConnectionPool(host='overpass.kumi.systems', port=443): Read timed out. (read timeout=120)
    504 en https://overpass.kumi.systems/api/i… reintentando
    504 en https://overpass-api.de/api/interpr… reintentando
    Error attempt 3 (https://overpass.openstreetmap.ru/a…): HTTPSConnectionPool(host='overpass.openstreetmap.ru', port=443): Max retries exceeded with url: /api/interpreter (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7ba9743f4c80>, 'Connection to overpass.openstreetmap.ru timed out. (connect timeout=120)'))
Resultado: {'road_length_km': None, 'poi_count': None, 'radius_m': 1000, 'error': True}


In [ ]:
# ── Cargar checkpoint si existe ────────────────────────────────────────────
checkpoint = {}
if Path(CHECKPOINT_PATH).exists():
    with open(CHECKPOINT_PATH) as f:
        checkpoint = json.load(f)
    n_ok  = sum(1 for v in checkpoint.values() if not v.get("error", True))
    n_err = sum(1 for v in checkpoint.values() if v.get("error", True))
    print(f"Checkpoint: {len(checkpoint)} zonas ({n_ok} OK, {n_err} errores — se reintentarán)")
else:
    print("Sin checkpoint — empezando desde zona 0.")

CHECKPOINT_EVERY = 10
SLEEP_BETWEEN   = 4.0  # segundos entre zonas (2 sub-queries por zona)

# Solo marcar como done las zonas SIN error (errores se reintentan)
done_ids = set(k for k, v in checkpoint.items() if not v.get("error", True))
results  = [v for v in checkpoint.values() if not v.get("error", True)]

zones_todo = [
    (i, row) for i, row in info.iterrows()
    if str(i) not in done_ids
]

print(f"Zonas restantes: {len(zones_todo)} / {len(info)}")
print(f"Tiempo estimado: ~{len(zones_todo) * (SLEEP_BETWEEN + 1.5) / 60:.0f} min")

for zone_id, row in tqdm(zones_todo, desc="Overpass spatial"):
    lat = row.get(LAT_COL, None)
    lon = row.get(LON_COL, None)

    if pd.isna(lat) or pd.isna(lon):
        entry = {"zone_id": zone_id, "lat": None, "lon": None,
                 "road_length_km": None, "poi_count": None, "error": True}
    else:
        spatial = fetch_spatial(float(lat), float(lon))
        entry   = {"zone_id": zone_id, "lat": float(lat), "lon": float(lon), **spatial}

    if not entry.get("error", True):
        results.append(entry)
        checkpoint[str(zone_id)] = entry
        if len(results) % CHECKPOINT_EVERY == 0:
            with open(CHECKPOINT_PATH, "w") as f:
                json.dump(checkpoint, f)
            print(f"  Checkpoint guardado — {len(results)} zonas OK")
    else:
        print(f"  Zona {zone_id}: error (se reintentará en próxima ejecución)")

    time.sleep(SLEEP_BETWEEN)

with open(CHECKPOINT_PATH, "w") as f:
    json.dump(checkpoint, f)

n_ok  = sum(1 for r in results if not r.get("error", False))
n_err = len(info) - n_ok
print(f"\nCompletado: {n_ok} zonas OK, {n_err} pendientes.")
if n_err > 0:
    print("Vuelve a ejecutar esta celda para reintentar las zonas fallidas.")

In [ ]:
# ── Guardar CSV ────────────────────────────────────────────────────────────
df_spatial = pd.DataFrame(results).sort_values("zone_id").reset_index(drop=True)

# Imputar errores con mediana de zonas correctas
for col in ["road_length_km", "poi_count"]:
    median_val = df_spatial[col].median()
    n_missing  = df_spatial[col].isna().sum()
    df_spatial[col] = df_spatial[col].fillna(median_val)
    if n_missing:
        print(f"  {col}: {n_missing} valores imputados con mediana={median_val:.2f}")

# Densidades normalizadas [0,1] para uso directo en prompts
for col in ["road_length_km", "poi_count"]:
    mn, mx = df_spatial[col].min(), df_spatial[col].max()
    df_spatial[f"{col}_norm"] = ((df_spatial[col] - mn) / (mx - mn + 1e-8)).round(4)

df_spatial.drop(columns=["error", "radius_m"], errors="ignore", inplace=True)
df_spatial.to_csv(OUTPUT_PATH, index=False)

print(f"\nGuardado: {OUTPUT_PATH}")
print(f"Shape:    {df_spatial.shape}")
print(f"\nStats:")
print(df_spatial[["road_length_km", "poi_count"]].describe().round(2))

if IN_COLAB:
    import shutil
    dst = f"{DATASETS_PATH}/spatial_features_shenzhen.csv"
    shutil.copy(OUTPUT_PATH, dst)
    print(f"\nCopiado a Drive: {dst}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Spatial Features — 247 zonas Shenzhen (radio 1 km)", fontsize=12)

axes[0].hist(df_spatial["road_length_km"], bins=30, color="#E74C3C", edgecolor="white", alpha=0.85)
axes[0].axvline(df_spatial["road_length_km"].median(), color="black", lw=1.5, ls="--",
                label=f"Mediana: {df_spatial['road_length_km'].median():.1f} km")
axes[0].set_xlabel("Road length (km)")
axes[0].set_ylabel("Frecuencia")
axes[0].set_title("Distribución Road Length")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].hist(df_spatial["poi_count"], bins=30, color="#2980B9", edgecolor="white", alpha=0.85)
axes[1].axvline(df_spatial["poi_count"].median(), color="black", lw=1.5, ls="--",
                label=f"Mediana: {df_spatial['poi_count'].median():.0f}")
axes[1].set_xlabel("POI count")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribución POI Density")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

axes[2].scatter(df_spatial["lon"], df_spatial["lat"],
                c=df_spatial["poi_count"], cmap="YlOrRd",
                s=40, alpha=0.8, edgecolors="grey", linewidths=0.3)
sm = plt.cm.ScalarMappable(cmap="YlOrRd",
     norm=plt.Normalize(df_spatial["poi_count"].min(), df_spatial["poi_count"].max()))
plt.colorbar(sm, ax=axes[2], label="POI count")
axes[2].set_xlabel("Longitud")
axes[2].set_ylabel("Latitud")
axes[2].set_title("Mapa POI Density — Shenzhen")
axes[2].grid(alpha=0.2)

plt.tight_layout()
plt.savefig("spatial_features_preview.png", dpi=130, bbox_inches="tight")
print("Preview guardado: spatial_features_preview.png")
plt.show()

print("\nTop 5 zonas por road length:")
print(df_spatial.nlargest(5, "road_length_km")[["zone_id", "lat", "lon", "road_length_km", "poi_count"]].to_string())
print("\nTop 5 zonas por POI density:")
print(df_spatial.nlargest(5, "poi_count")[["zone_id", "lat", "lon", "road_length_km", "poi_count"]].to_string())